In [24]:
import os
import json

from dataclasses import dataclass
from pydantic import BaseModel
from typing import Literal

from dotenv import load_dotenv

load_dotenv()

@dataclass(frozen=True)
class Provider:
    """One Provider described as pure DATA."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS=[
    Provider("Groq", "GROQ_API_KEY", True, None, "openai/gpt-oss-20b"),
]


def select_provider()->Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return provider
    expected=", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set. Add one of {expected} to your .env file")


def build_client(provider: Provider):
    from groq import Groq

    api_key=os.environ[provider.env_var]
    if provider.base_url is None:
        return Groq(api_key=api_key)
    return Groq(api_key=api_key, base_url=provider.base_url)


def llm_reply(prompt: str,*,max_tokens: int=300)->str:
    """Send one user prompt; return the assistant's text."""

    provider=select_provider()
    client=build_client(provider)

    result=client.chat.completions.create(
        model=provider.model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return result.choices[0].message.content

In [25]:
class SentimentResult(BaseModel):
    """Sentiment Result from LLM"""

    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float
    reason: str

In [26]:
def validate_sentiment_reply(raw_reply: str)->SentimentResult | str:
    try:
        parsed=json.loads(raw_reply)
        return SentimentResult(**parsed)
    except json.JSONDecodeError:
        return f"Invalid JSON format: {raw_reply}"

In [27]:
SYSTEM_PROMPT="""
You are a product review sentiment analyst.

Read each review the user sends and classify its sentiment.
We should also attach a confidence score for the sentiment between 0.0 to 1.0 and a short explanation for the sentiment as a reason.

For each review respond with only a JSON object - no other text or comments - in exactly the following shape:

{
    "sentiment": "positive" | "negative" | "neutral",
    "confidence": 0.0 to 1.0,
    "reason": "Short explanation for the sentiment"
}
""".strip()

In [28]:
def analyze_review(incoming_review: str)->str:
    """Send review to LLM and ask it to classify the sentiment"""

    provider=select_provider()
    client=build_client(provider)

    result=client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": incoming_review
            }
        ]
    )

    return result.choices[0].message.content.strip()

In [29]:
REVIEWS=[
    "Absolutely loved the headphones, great sounds and battery lasts all day!",
    "Arrived broken and customer service never replied. Very disappointed.",
    "It works. Nothing special, nothing terrible."
]

In [30]:
review1=analyze_review(REVIEWS[0])
result=validate_sentiment_reply(review1)
print(result)
print(type(result))

sentiment='positive' confidence=0.95 reason='User expresses clear enthusiasm and satisfaction with sound quality and battery life.'
<class '__main__.SentimentResult'>


In [31]:
review2=analyze_review(REVIEWS[1])
result=validate_sentiment_reply(review2)
print(result)
print(type(result))

sentiment='negative' confidence=0.98 reason='Review expresses dissatisfaction due to broken product and lack of customer service response'
<class '__main__.SentimentResult'>


In [32]:
review3=analyze_review(REVIEWS[2])
result=validate_sentiment_reply(review3)
print(result)
print(type(result))

sentiment='neutral' confidence=0.7 reason='The review mentions it works but has neither special features nor terribleness, indicating a neutral stance.'
<class '__main__.SentimentResult'>
